In [1]:
import requests
import pandas as pd
import dotenv
import os
import time

dotenv.load_dotenv()

API_KEY = os.getenv("RiotAPIKey")

headers = {
    "X-Riot-Token": API_KEY
}

# =====================================================
# account -> puuid
# =====================================================

def get_puuid(game_name, tag_line):

    url = (
        f"https://asia.api.riotgames.com"
        f"/riot/account/v1/accounts"
        f"/by-riot-id/{game_name}/{tag_line}"
    )

    r = requests.get(
        url,
        headers=headers
    )

    print(f"\n[{game_name}#{tag_line}]")
    print("STATUS:", r.status_code)

    if r.status_code != 200:
        print(r.text)
        return None

    return r.json()


# =====================================================
# 테스트할 계정들
# =====================================================

accounts = [

    ("오 너", "111"),

    ("T1 Oner", "KR1"),

    ("2887168037291584", "KR1"),

    ("에휴휴휴휴휴", "KR1")

]

# =====================================================
# puuid 수집
# =====================================================

all_accounts = []

for game_name, tag_line in accounts:

    data = get_puuid(
        game_name,
        tag_line
    )

    if data is None:
        continue

    puuid = data.get("puuid")

    print("PUUID:", puuid)

    all_accounts.append({

        "riot_id": f"{game_name}#{tag_line}",

        "puuid": puuid

    })

    time.sleep(1)

# =====================================================
# DataFrame 확인
# =====================================================

accounts_df = pd.DataFrame(all_accounts)

print("\n==============================")
print(accounts_df)
print("==============================")


# =====================================================
# match id 조회 함수
# =====================================================

def get_recent_matches(
    puuid,
    count=5
):

    url = (
        f"https://asia.api.riotgames.com"
        f"/lol/match/v5/matches/by-puuid/{puuid}/ids"
        f"?start=0"
        f"&count={count}"
    )

    r = requests.get(
        url,
        headers=headers
    )

    print("\nMATCH STATUS:", r.status_code)

    if r.status_code != 200:
        print(r.text)
        return []

    return r.json()


# =====================================================
# match detail 조회 함수
# =====================================================

def get_match_detail(match_id):

    url = (
        f"https://asia.api.riotgames.com"
        f"/lol/match/v5/matches/{match_id}"
    )

    r = requests.get(
        url,
        headers=headers
    )

    print("DETAIL STATUS:", r.status_code)

    if r.status_code != 200:
        print(r.text)
        return None

    return r.json()


# =====================================================
# 각 계정 테스트
# =====================================================

for idx, row in accounts_df.iterrows():

    riot_id = row["riot_id"]
    puuid = row["puuid"]

    print("\n================================================")
    print("ACCOUNT:", riot_id)
    print("================================================")

    match_ids = get_recent_matches(
        puuid,
        count=5
    )

    print("\nMATCH IDS")
    print(match_ids)

    if len(match_ids) == 0:
        continue

    sample_match = match_ids[0]

    detail = get_match_detail(sample_match)

    if detail is None:
        continue

    info = detail["info"]

    print("\n========================")
    print("MATCH INFO")
    print("========================")

    print("MATCH ID:", sample_match)

    print("GAME VERSION:")
    print(info["gameVersion"])

    print("\nQUEUE ID:")
    print(info["queueId"])

    print("\nGAME CREATION:")
    print(info["gameCreation"])

    from datetime import datetime

    print(
        datetime.fromtimestamp(
            info["gameCreation"] / 1000
        )
    )

    print("\nPARTICIPANTS:")

    for p in info["participants"]:

        print(
            p["riotIdGameName"],
            "#",
            p["riotIdTagline"]
        )

    time.sleep(2)


[오 너#111]
STATUS: 200
PUUID: OIYXiaBVLwTG3o2REcHVFuO6VgqilutuTnqY04Waw_jYPlY7ZzBzDMpnmsc6UEdXA3cZQGKRpm94lA

[T1 Oner#KR1]
STATUS: 200
PUUID: SX7Kqzc8v8ZK6_iBQKM_oaoSVCKhpryONr5Y5cveKUS1gXSTZimu4_DozAgkP89rZZwG3wHC3Oqjeg

[2887168037291584#KR1]
STATUS: 200
PUUID: x0s_LmsP2wycyYNRtHMAz_6pyhEA4uDTGgu326mJ0Ju6jRR6gj7wG_tfvZ3fQ3FlwCvYBnU4DJjKeg

[에휴휴휴휴휴#KR1]
STATUS: 200
PUUID: ig21Klng6ZL-BBAfo3340GgnWLoIJixu5FAt3iRb1ctVtSuGJrz82AzKG0IuCCLlQ_hEiqAu-FkIAQ

                riot_id                                              puuid
0               오 너#111  OIYXiaBVLwTG3o2REcHVFuO6VgqilutuTnqY04Waw_jYPl...
1           T1 Oner#KR1  SX7Kqzc8v8ZK6_iBQKM_oaoSVCKhpryONr5Y5cveKUS1gX...
2  2887168037291584#KR1  x0s_LmsP2wycyYNRtHMAz_6pyhEA4uDTGgu326mJ0Ju6jR...
3            에휴휴휴휴휴#KR1  ig21Klng6ZL-BBAfo3340GgnWLoIJixu5FAt3iRb1ctVtS...

ACCOUNT: 오 너#111

MATCH STATUS: 200

MATCH IDS
['KR_8234568482', 'KR_8234540360', 'KR_8234508008', 'KR_8233384738', 'KR_8233335810']
DETAIL STATUS: 200

MATCH INFO
MAT

In [11]:
import requests
import dotenv
import os
import time
import pandas as pd

dotenv.load_dotenv()

API_KEY = os.getenv("RiotAPIKey")

headers = {
    "X-Riot-Token": API_KEY
}

# =====================================================
# 테스트할 puuid
# =====================================================

puuid = "lLj5ZFE5BYnVXkJJim-UKSfy5NO0UkyByxYHvRcLIe1246rN_dnQUcW0WIyR5McqzgruVXODuiQsOg"

# =====================================================
# 최신 match id 1000개 수집
# =====================================================

all_match_ids = []

TOTAL_COUNT = 10000

for start in range(0, TOTAL_COUNT, 100):

    url = (
        f"https://asia.api.riotgames.com"
        f"/lol/match/v5/matches/by-puuid/{puuid}/ids"
        f"?start={start}"
        f"&count=100"
    )

    r = requests.get(
        url,
        headers=headers
    )

    print("\n===================================")
    print(f"START: {start}")
    print("STATUS:", r.status_code)

    if r.status_code != 200:

        print(r.text)
        break

    match_ids = r.json()

    print("COUNT:", len(match_ids))

    # 더 이상 없으면 종료
    if len(match_ids) == 0:

        print("NO MORE MATCHES")
        break

    all_match_ids.extend(match_ids)

    # 중복 제거
    all_match_ids = list(
        dict.fromkeys(all_match_ids)
    )

    time.sleep(1.2)

# =====================================================
# 결과 확인
# =====================================================

print("\n===================================")
print("FINAL RESULT")
print("===================================")

print("TOTAL MATCHES:", len(all_match_ids))

print("\nFIRST 10:")
print(all_match_ids[:10])

print("\nLAST 10:")
print(all_match_ids[-10:])

# =====================================================
# 가장 오래된 match detail 확인
# =====================================================

if len(all_match_ids) > 0:

    oldest_match = all_match_ids[-1]

    print("\nOLDEST MATCH:")
    print(oldest_match)

    detail_url = (
        f"https://asia.api.riotgames.com"
        f"/lol/match/v5/matches/{oldest_match}"
    )

    r2 = requests.get(
        detail_url,
        headers=headers
    )

    detail = r2.json()

    info = detail["info"]

    from datetime import datetime

    print("\n==========================")
    print("OLDEST MATCH INFO")
    print("==========================")

    print("GAME VERSION:")
    print(info["gameVersion"])

    print("\nQUEUE ID:")
    print(info["queueId"])

    print("\nGAME DATE:")

    print(
        datetime.fromtimestamp(
            info["gameCreation"] / 1000
        )
    )


START: 0
STATUS: 200
COUNT: 100

START: 100
STATUS: 200
COUNT: 100

START: 200
STATUS: 200
COUNT: 100

START: 300
STATUS: 200
COUNT: 100

START: 400
STATUS: 200
COUNT: 71

START: 500
STATUS: 200
COUNT: 0
NO MORE MATCHES

FINAL RESULT
TOTAL MATCHES: 471

FIRST 10:
['KR_8234596600', 'KR_8234586157', 'KR_8234562395', 'KR_8233424865', 'KR_8233397494', 'KR_8233249193', 'KR_8232977799', 'KR_8232930667', 'KR_8232826677', 'KR_8228917641']

LAST 10:
['KR_8013785647', 'KR_8012797809', 'KR_8012729639', 'KR_8012668721', 'KR_8012596643', 'KR_8011153592', 'KR_8011086853', 'KR_8011016871', 'KR_8010975689', 'KR_8009922403']

OLDEST MATCH:
KR_8009922403

OLDEST MATCH INFO
GAME VERSION:
15.24.734.7485

QUEUE ID:
420

GAME DATE:
2026-01-05 16:09:17.703000


In [10]:
import requests
import dotenv
import os

dotenv.load_dotenv()

API_KEY = os.getenv("RiotAPIKey")

headers = {
    "X-Riot-Token": API_KEY
}

# =========================================
# 본인 계정 입력
# =========================================

game_name = "Hide on bush"
tag_line = "KR1"

# 예시:
# game_name = "Hide on bush"
# tag_line = "KR1"

# =========================================
# Riot Account API
# =========================================

url = (
    f"https://asia.api.riotgames.com"
    f"/riot/account/v1/accounts"
    f"/by-riot-id/{game_name}/{tag_line}"
)

r = requests.get(
    url,
    headers=headers
)

print("STATUS:", r.status_code)

data = r.json()

print("\nRESULT")
print(data)

STATUS: 200

RESULT
{'puuid': 'lLj5ZFE5BYnVXkJJim-UKSfy5NO0UkyByxYHvRcLIe1246rN_dnQUcW0WIyR5McqzgruVXODuiQsOg', 'gameName': 'Hide on bush', 'tagLine': 'KR1'}
